# 03 - Evaluation：MLflow GenAI 評估框架

本筆記本示範如何使用 MLflow 3.x GenAI evaluate 進行模型評估，
包含內建 scorers、自定義 scorers 與 LLM-as-a-Judge。

## 簡介

MLflow GenAI 提供完整的評估框架：

- **`mlflow.genai.evaluate()`**：核心評估 API，自動執行 predict_fn 並套用 scorers
- **LLM Judge**：使用內部 LLMService 建立 LLM-as-a-Judge（取代 MLflow 內建的 `Correctness`、`RelevanceToQuery`、`Safety`）
- **`@scorer` decorator**：快速建立自定義 rule-based scorer
- **`run_evaluation()`**：本框架封裝的便捷函式

> 所有 LLM 評估均使用內部 LLMService，不依賴外部 LLM API key。

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

import mlflow
from app.utils.config import init_config
from app.logger import init_mlflow

cfg = init_config()
init_mlflow(cfg)

print("設定與 MLflow 初始化完成。")

## 準備評估資料

MLflow GenAI evaluate 的資料格式：

```python
{
    "inputs": {"question": "..."},          # predict_fn 的輸入
    "expectations": {"expected_facts": [...]}, # 預期結果（供 scorer 使用）
}
```

In [ ]:
eval_data = [
    {
        "inputs": {"question": "台灣的首都是哪裡？"},
        "expectations": {
            "expected_facts": ["台北"],
            "expected": "台北",
            "keywords": "台北,首都",
        },
    },
    {
        "inputs": {"question": "什麼是 Python？"},
        "expectations": {
            "expected_facts": ["程式語言", "高階語言"],
            "expected": "Python 是一種高階程式語言",
            "keywords": "程式語言,Python",
        },
    },
    {
        "inputs": {"question": "1 + 1 = ?"},
        "expectations": {
            "expected_facts": ["2"],
            "expected": "2",
            "keywords": "2",
        },
    },
]

print(f"準備了 {len(eval_data)} 筆評估資料。")

## 定義 Predict Function

`predict_fn` 接受 eval_data 中 `inputs` 的 key-value 作為參數，回傳字串。
此處使用 mock，實際可替換為呼叫 LLM 的函式。

In [ ]:
MOCK_ANSWERS = {
    "台灣的首都是哪裡？": "台灣的首都是台北。",
    "什麼是 Python？": "Python 是一種高階程式語言，廣泛用於資料科學、Web 開發等領域。",
    "1 + 1 = ?": "2",
}


@mlflow.trace
def my_app(question: str) -> str:
    """模擬 LLM 應用。"""
    return MOCK_ANSWERS.get(question, "我不知道答案。")


# 快速測試
print(my_app("台灣的首都是哪裡？"))

## 使用 LLM Judge 取代內建 Scorers

本框架提供基於內部 LLMService 的 LLM Judge，取代 MLflow 內建的 `Correctness`、`RelevanceToQuery`、`Safety`（後者需要外部 LLM API key）。

| Judge | 取代 | 用途 |
|-------|------|------|
| `create_correctness_judge()` | `Correctness()` | 檢查是否包含預期事實 |
| `create_relevance_judge()` | `RelevanceToQuery()` | 回答與問題的相關性 |
| `create_safety_judge()` | `Safety()` | 安全性檢查 |

> 這些 Judge 使用內部 LLMService，無需設定外部 LLM API key。

In [ ]:
from app.evaluator.scorers import (
    create_correctness_judge,
    create_relevance_judge,
    create_safety_judge,
)

# 使用框架封裝的 run_evaluation
from app.evaluator import run_evaluation

# 使用內部 LLMService 的 LLM Judge（取代 MLflow 內建的 Correctness / RelevanceToQuery）
try:
    correctness_judge = create_correctness_judge()
    relevance_judge = create_relevance_judge()

    results = run_evaluation(
        predict_fn=my_app,
        eval_data=eval_data,
        scorers=[correctness_judge, relevance_judge],
        run_name="builtin_replacement_judges_demo",
    )
    print("=== 評估結果 ===")
    print(results.tables["eval_results"])
except Exception as e:
    print(f"[提示] LLM Judge 需要可用的 LLM 服務：{e}")
    print("請往下查看 rule-based scorer 的範例。")

## 使用自定義 Rule-based Scorers

使用 `@scorer` decorator 建立不需要 LLM 的 rule-based scorer。
本框架在 `app/evaluator/scorers.py` 中預定義了幾個常用的：

In [ ]:
from app.evaluator.scorers import (
    response_not_empty,
    response_length_check,
    exact_match,
    contains_keywords,
)

# 使用 rule-based scorers（不需要 LLM API key）
results = run_evaluation(
    predict_fn=my_app,
    eval_data=eval_data,
    scorers=[response_not_empty, response_length_check, exact_match, contains_keywords],
    run_name="rule_based_scorers_demo",
)

print("=== Rule-based Scorer 評估結果 ===")
print(results.tables["eval_results"])

## 建立自己的 Scorer

使用 `@scorer` decorator，回傳 `bool`、`float` 或 `Feedback` 物件。

In [ ]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback


@scorer
def is_chinese(outputs: str) -> Feedback:
    """檢查輸出是否包含中文字元。"""
    chinese_chars = sum(1 for c in outputs if '\u4e00' <= c <= '\u9fff')
    total_chars = len(outputs)
    if total_chars == 0:
        return Feedback(value=0.0, rationale="Empty output")
    ratio = chinese_chars / total_chars
    return Feedback(
        value=ratio,
        rationale=f"{chinese_chars}/{total_chars} characters are Chinese ({ratio:.1%})",
    )


# 混合使用自定義與預定義 scorer
results = run_evaluation(
    predict_fn=my_app,
    eval_data=eval_data,
    scorers=[response_not_empty, is_chinese, contains_keywords],
    run_name="custom_scorer_demo",
)

print("=== 含自定義 Scorer 的評估結果 ===")
print(results.tables["eval_results"])

## LLM-as-a-Judge

使用 `create_llm_judge()` 建立以 LLM 為評判的 scorer。
底層使用 LLMService，自動從 `llm_config.yaml` 載入設定。

> **注意**：需要可用的 LLM 服務。

In [ ]:
from app.evaluator.scorers import create_quality_judge, create_tone_judge

# 建立 LLM judge（自動從 llm_config.yaml 載入設定，使用 litellm）
try:
    quality_judge = create_quality_judge()
    tone_judge = create_tone_judge()

    results = run_evaluation(
        predict_fn=my_app,
        eval_data=eval_data,
        scorers=[quality_judge, tone_judge],
        run_name="llm_judge_demo",
    )

    print("=== LLM Judge 評估結果 ===")
    print(results.tables["eval_results"])
except Exception as e:
    print(f"[提示] LLM Judge 需要可用的 LLM 服務：{e}")

## 對已有結果進行評估（Trace Evaluation）

如果已有預測結果，可以直接評估而不需要 predict_fn。

In [ ]:
from app.evaluator import run_trace_evaluation

# 已有預測結果的資料
existing_data = [
    {
        "inputs": {"question": "台灣的首都是哪裡？"},
        "outputs": "台灣的首都是台北。",
        "expectations": {"expected": "台北", "keywords": "台北"},
    },
    {
        "inputs": {"question": "1 + 1 = ?"},
        "outputs": "答案是 2。",
        "expectations": {"expected": "2", "keywords": "2"},
    },
]

results = run_trace_evaluation(
    data=existing_data,
    scorers=[response_not_empty, contains_keywords],
    run_name="trace_eval_demo",
)

print("=== Trace Evaluation 結果 ===")
print(results.tables["eval_results"])

## 小結

| 功能 | API | 說明 |
|------|-----|------|
| 執行評估 | `run_evaluation()` / `mlflow.genai.evaluate()` | 自動執行 predict_fn + scorers |
| 正確性 Judge | `create_correctness_judge()` | 使用 LLMService 檢查預期事實 |
| 相關性 Judge | `create_relevance_judge()` | 使用 LLMService 評估回答相關性 |
| 安全性 Judge | `create_safety_judge()` | 使用 LLMService 檢查安全性 |
| Rule-based | `@scorer` decorator | 不需 LLM，純邏輯評分 |
| LLM Judge | `create_llm_judge()` | 使用 LLMService 的自訂 LLM 評判 |
| 品質 Judge | `create_quality_judge()` | 使用 LLMService 評估回答品質 |
| 語調 Judge | `create_tone_judge()` | 使用 LLMService 評估專業語調 |
| Trace 評估 | `run_trace_evaluation()` | 對已有結果評估 |

> 所有 LLM Judge 均使用內部 LLMService，不依賴外部 LLM API。

### 後續步驟

- `04_prompt_management.ipynb`：Prompt 註冊、版本管理與自動優化
- 啟動 `mlflow ui --port 5000` 查看評估結果與比較
- 設定 CI/CD 自動化評估流程